In [58]:
import requests
import json
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv

load_dotenv('secrets.env')

True

In [59]:
# Google Authentication
PROJECT_ID = os.getenv('PROJECT_ID')
DATASET_ID = os.getenv('DATASET_ID')
TABLE_ID = 'vend_products'

# Replace with the path to your service account key file
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')


In [60]:
# Initialize BigQuery client
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

# Define the BigQuery table reference
dataset_ref = client.dataset(DATASET_ID)
table_ref = dataset_ref.table(TABLE_ID)

In [61]:
schema = [
    bigquery.SchemaField("id", "STRING"),
    bigquery.SchemaField("source_id", "STRING"),
    bigquery.SchemaField("source_variant_id", "STRING"),
    bigquery.SchemaField("variant_parent_id", "STRING"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("variant_name", "STRING"),
    bigquery.SchemaField("handle", "STRING"),
    bigquery.SchemaField("sku", "STRING"),
    bigquery.SchemaField("supplier_code", "STRING"),
    bigquery.SchemaField("active", "BOOLEAN"),
    bigquery.SchemaField("ecwid_enabled_webstore", "BOOLEAN"),
    bigquery.SchemaField("has_inventory", "BOOLEAN"),
    bigquery.SchemaField("is_composite", "BOOLEAN"),
    bigquery.SchemaField("description", "STRING"),
    bigquery.SchemaField("image_url", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
    bigquery.SchemaField("updated_at", "TIMESTAMP"),
    bigquery.SchemaField("deleted_at", "TIMESTAMP"),
    bigquery.SchemaField("source", "STRING"),
    bigquery.SchemaField("account_code", "STRING"),
    bigquery.SchemaField("account_code_purchase", "STRING"),
    bigquery.SchemaField("supply_price", "FLOAT"),
    bigquery.SchemaField("version", "INTEGER"),
    bigquery.SchemaField("type", "RECORD", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("name", "STRING"),
        bigquery.SchemaField("deleted_at", "TIMESTAMP"),
        bigquery.SchemaField("version", "INTEGER"),
    ]),
    bigquery.SchemaField("product_category", "RECORD", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("name", "STRING"),
        bigquery.SchemaField("leaf_category", "BOOLEAN"),
        bigquery.SchemaField("category_path", "RECORD", mode="REPEATED", fields=[
            bigquery.SchemaField("id", "STRING"),
            bigquery.SchemaField("name", "STRING"),
        ]),
    ]),
    bigquery.SchemaField("supplier", "RECORD", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("name", "STRING"),
        bigquery.SchemaField("source", "STRING"),
        bigquery.SchemaField("description", "STRING"),
        bigquery.SchemaField("deleted_at", "TIMESTAMP"),
        bigquery.SchemaField("version", "INTEGER"),
    ]),
    bigquery.SchemaField("brand", "RECORD", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("name", "STRING"),
        bigquery.SchemaField("description", "STRING"),
        bigquery.SchemaField("deleted_at", "TIMESTAMP"),
        bigquery.SchemaField("version", "INTEGER"),
    ]),
    bigquery.SchemaField("variant_options", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("name", "STRING"),
        bigquery.SchemaField("value", "STRING"),
    ]),
    bigquery.SchemaField("categories", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("name", "STRING"),
        bigquery.SchemaField("deleted_at", "TIMESTAMP"),
        bigquery.SchemaField("version", "INTEGER"),
    ]),
    bigquery.SchemaField("has_variants", "BOOLEAN"),
    bigquery.SchemaField("variant_count", "INTEGER"),
    bigquery.SchemaField("button_order", "INTEGER"),
    bigquery.SchemaField("price_including_tax", "FLOAT"),
    bigquery.SchemaField("price_excluding_tax", "FLOAT"),
    bigquery.SchemaField("loyalty_amount", "FLOAT"),
    bigquery.SchemaField("product_codes", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("type", "STRING"),
        bigquery.SchemaField("code", "STRING"),
    ]),
    bigquery.SchemaField("product_suppliers", "RECORD", mode="REPEATED", fields=[
        bigquery.SchemaField("id", "STRING"),
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("supplier_id", "STRING"),
        bigquery.SchemaField("supplier_name", "STRING"),
        bigquery.SchemaField("code", "STRING"),
        bigquery.SchemaField("price", "FLOAT"),
    ]),
    bigquery.SchemaField("packaging", "RECORD", fields=[
        bigquery.SchemaField("made_from", "STRING", mode="REPEATED"),
        bigquery.SchemaField("breaks_into", "STRING", mode="REPEATED"),
    ]),
    bigquery.SchemaField("weight", "FLOAT"),
    bigquery.SchemaField("weight_unit", "STRING"),
    bigquery.SchemaField("length", "FLOAT"),
    bigquery.SchemaField("width", "FLOAT"),
    bigquery.SchemaField("height", "FLOAT"),
    bigquery.SchemaField("dimensions_unit", "STRING"),
    bigquery.SchemaField("is_active", "BOOLEAN"),
    bigquery.SchemaField("image_thumbnail_url", "STRING"),
    bigquery.SchemaField("product_type_id", "STRING"),
    bigquery.SchemaField("supplier_id", "STRING"),
    bigquery.SchemaField("brand_id", "STRING"),
    bigquery.SchemaField("tag_ids", "STRING", mode="REPEATED"),
]

In [62]:
# Create the table if it doesn't exist
table = bigquery.Table(table_ref, schema=schema)
table = client.create_table(table, exists_ok=True)

In [63]:
# Function to fetch data from Vend API
def fetch_vend_data(url, headers, params=None):
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()


In [64]:
# Function to load data into BigQuery
def load_data_to_bigquery(client, table_ref, rows_to_insert):
    errors = client.insert_rows_json(table_ref, rows_to_insert)
    if errors:
        print(f"Encountered errors while inserting rows: {errors}")
    else:
        print("Data successfully inserted into BigQuery.")


In [65]:
# Vend API details
vend_url = "https://ashcorp.retail.lightspeed.app/api/2.0/products"
vend_headers = {
    "accept": "application/json",
    "authorization": f"Bearer {os.getenv('LIGHTSPEED_ACCESS_TOKEN')}"
}

In [67]:
# Pagination parameters
after = 0

# Extract, Transform, Load (ETL) process
while True:
    # Extract data from Vend API
    params = {"after": after}
    data = fetch_vend_data(vend_url, vend_headers, params)

    # Transform data (if needed)
    rows_to_insert = data["data"]

    # Remove 'images', 'skuImages', and 'attributes' fields from the JSON output
    for row in rows_to_insert:
        if 'images' in row:
            del row['images']
        if 'skuImages' in row:
            del row['skuImages']
        if 'attributes' in row:
            del row['attributes']

    
    # Load data into BigQuery
    load_data_to_bigquery(client, table_ref, rows_to_insert)
    
    # Get the max version number for the next request
    after = data["version"]["max"]

    
    # Check if the data collection is empty
    if not data["data"]:
        break

print("ETL process completed successfully.")

Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.
Data successfully inserted into BigQuery.


BadRequest: 400 POST https://bigquery.googleapis.com/bigquery/v2/projects/eastern-rider-451408-r3/datasets/vend_raw/tables/vend_products/insertAll?prettyPrint=false: No rows present in the request.